In [ ]:
# Line-in -> Noise Gate -> Overdrive -> Distortion -> EQ -> Reverb -> output jack
import audio_lab_pynq as aud
from IPython.display import display

ol = aud.AudioLabOverlay()

def configure_audio_path(volume=0xE7):
    # Line-in AUX -> ADC
    ol.codec.R4_RECORD_MIXER_LEFT_CONTROL_0 = 0x01
    ol.codec.R5_RECORD_MIXER_LEFT_CONTROL_1 = 0x05
    ol.codec.R6_RECORD_MIXER_RIGHT_CONTROL_0 = 0x01
    ol.codec.R7_RECORD_MIXER_RIGHT_CONTROL_1 = 0x05
    ol.codec.R19_ADC_CONTROL = 0x03

    # FPGA serial playback -> DAC -> playback mixers
    ol.codec.R58_SERIAL_INPUT_ROUTE_CONTROL = 0x01
    ol.codec.R59_SERIAL_OUTPUT_ROUTE_CONTROL = 0x01
    ol.codec.R36_DAC_CONTROL_0 = 0x03
    ol.codec.R22_PLAYBACK_MIXER_LEFT_CONTROL_0 = 0x21
    ol.codec.R24_PLAYBACK_MIXER_RIGHT_CONTROL_0 = 0x41

    # PYNQ-Z2 output jack path
    ol.codec.R26_PLAYBACK_LR_MIXER_LEFT_LINE_OUTPUT_CONTROL = 0x03
    ol.codec.R27_PLAYBACK_LR_MIXER_RIGHT_LINE_OUTPUT_CONTROL = 0x09
    ol.codec.R31_PLAYBACK_LINE_OUTPUT_LEFT_VOLUME_CONTROL = volume
    ol.codec.R32_PLAYBACK_LINE_OUTPUT_RIGHT_VOLUME_CONTROL = volume

    # Keep headphone output path enabled as well.
    ol.codec.R29_PLAYBACK_HEADPHONE_LEFT_VOLUME_CONTROL = volume
    ol.codec.R30_PLAYBACK_HEADPHONE_RIGHT_VOLUME_CONTROL = volume
    ol.codec.R35_PLAYBACK_POWER_MANAGEMENT = 0x03

    # Codec clocks/DSP
    ol.codec.R61_DSP_ENABLE = 0x01
    ol.codec.R62_DSP_RUN = 0x01
    ol.codec.R65_CLOCK_ENABLE_0 = 0x7F
    ol.codec.R66_CLOCK_ENABLE_1 = 0x03

configure_audio_path()

def apply_effects(
    noise_gate_on=False, noise_gate_threshold=8,
    overdrive_on=False, overdrive_tone=65, overdrive_level=100, overdrive_drive=30,
    distortion_on=False, distortion_tone=65, distortion_level=100, distortion=25,
    eq_on=False, eq_low=100, eq_mid=100, eq_high=100,
    reverb_on=False, reverb_decay=30, reverb_tone=65, reverb_mix=20,
):
    words = ol.set_guitar_effects(
        noise_gate_on=noise_gate_on,
        noise_gate_threshold=noise_gate_threshold,
        overdrive_on=overdrive_on,
        overdrive_tone=overdrive_tone,
        overdrive_level=overdrive_level,
        overdrive_drive=overdrive_drive,
        distortion_on=distortion_on,
        distortion_tone=distortion_tone,
        distortion_level=distortion_level,
        distortion=distortion,
        eq_on=eq_on,
        eq_low=eq_low,
        eq_mid=eq_mid,
        eq_high=eq_high,
        reverb_on=reverb_on,
        reverb_decay=reverb_decay,
        reverb_tone=reverb_tone,
        reverb_mix=reverb_mix,
    )
    active = [
        name for name, on in [
            ("Noise Gate", noise_gate_on),
            ("Overdrive", overdrive_on),
            ("Distortion", distortion_on),
            ("EQ", eq_on),
            ("Reverb", reverb_on),
        ] if on
    ]
    print("Route:", " -> ".join(["Line-in"] + active + ["Output"]) if active else "Line-in -> Output")
    for key in ["gate", "overdrive", "distortion", "eq", "reverb"]:
        print("%10s: 0x%08x" % (key, words[key]))
    return words

try:
    import ipywidgets as widgets

    def cb(label, value=False):
        return widgets.Checkbox(value=value, description=label, indent=False)

    def slider(label, value, minimum=0, maximum=100):
        return widgets.IntSlider(
            value=value,
            min=minimum,
            max=maximum,
            step=1,
            description=label,
            continuous_update=False,
            style={"description_width": "95px"},
            layout=widgets.Layout(width="520px"),
        )

    gate_on = cb("Noise Gate", False)
    gate_threshold = slider("THRESHOLD", 8)

    od_on = cb("Overdrive", False)
    od_tone = slider("TONE", 65)
    od_level = slider("LEVEL", 100, 0, 200)
    od_drive = slider("DRIVE", 30)

    dist_on = cb("Distortion", False)
    dist_tone = slider("TONE", 65)
    dist_level = slider("LEVEL", 100, 0, 200)
    dist_amount = slider("DISTORTION", 25)

    eq_on = cb("EQ", False)
    eq_low = slider("LOW", 100, 0, 200)
    eq_mid = slider("MID", 100, 0, 200)
    eq_high = slider("HIGH", 100, 0, 200)


    reverb_on = cb("Reverb", False)
    reverb_decay = slider("Decay", 30)
    reverb_tone = slider("tone", 65)
    reverb_mix = slider("mix", 20)

    output = widgets.Output()

    sections = [
        widgets.VBox([gate_on, gate_threshold]),
        widgets.VBox([od_on, od_tone, od_level, od_drive]),
        widgets.VBox([dist_on, dist_tone, dist_level, dist_amount]),
        widgets.VBox([eq_on, eq_low, eq_mid, eq_high]),
        widgets.VBox([reverb_on, reverb_decay, reverb_tone, reverb_mix]),
    ]
    names = ["1 Noise Gate", "2 Overdrive", "3 Distortion", "4 EQ", "5 Reverb"]
    accordion = widgets.Accordion(children=sections)
    for i, name in enumerate(names):
        accordion.set_title(i, name)

    all_controls = [
        gate_on, gate_threshold,
        od_on, od_tone, od_level, od_drive,
        dist_on, dist_tone, dist_level, dist_amount,
        eq_on, eq_low, eq_mid, eq_high,
        reverb_on, reverb_decay, reverb_tone, reverb_mix,
    ]

    def update(_=None):
        with output:
            output.clear_output(wait=True)
            apply_effects(
                noise_gate_on=gate_on.value,
                noise_gate_threshold=gate_threshold.value,
                overdrive_on=od_on.value,
                overdrive_tone=od_tone.value,
                overdrive_level=od_level.value,
                overdrive_drive=od_drive.value,
                distortion_on=dist_on.value,
                distortion_tone=dist_tone.value,
                distortion_level=dist_level.value,
                distortion=dist_amount.value,
                eq_on=eq_on.value,
                eq_low=eq_low.value,
                eq_mid=eq_mid.value,
                eq_high=eq_high.value,
                reverb_on=reverb_on.value,
                reverb_decay=reverb_decay.value,
                reverb_tone=reverb_tone.value,
                reverb_mix=reverb_mix.value,
            )

    for control in all_controls:
        control.observe(update, names="value")

    display(widgets.VBox([accordion, output]))
    update()
except Exception as exc:
    print("ipywidgets is not available; use apply_effects(...) manually.")
    print("Example: apply_effects(overdrive_on=True, overdrive_drive=45, reverb_on=True, reverb_mix=20)")
    print("Widget error:", exc)
    apply_effects()
